In [ ]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time
import json
import pandas pd

In [ ]:
load_dotenv()  
def get_token(secret: str, url: str = "https://dep.simondg.com/auth/login"):
    response = requests.post(url, json={"secret": secret})
    response.raise_for_status() # Raises an error if the request fails
    return response.json()["access_token"]


def get_students_by_subgroup(token: str, subgroup_id: int):
    url = f"https://dep.simondg.com/students/{subgroup_id}"
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raises an error if the request fails
    
    data = response.json()
    
    return data


secret = os.getenv("SECRET")
token = get_token(secret)



In [ ]:
print(get_students_by_subgroup(token, 5879763 ) )  #test

In [ ]:
# Your subgroup file
subgroup_file = "../data/unique_classgroups.csv"
df_subgroups = pd.read_csv(subgroup_file, header=None, names=['subgroup_id'])

all_data_json = {"students": []}

MAX_RETRIES = 5
INITIAL_WAIT = 10  # seconds

def fetch_students_with_retry(subgroup_id, token):
    """Fetch students with retry and skip for server errors."""
    retries = 0
    wait = INITIAL_WAIT
    while retries < MAX_RETRIES:
        try:
            students = get_students_by_subgroup(token, str(subgroup_id))
            return students
        except requests.exceptions.HTTPError as e:
            status = e.response.status_code
            if status == 429:
                print(f"⚠️ Rate limit hit for {subgroup_id}, waiting {wait}s...")
                time.sleep(wait)
                retries += 1
                wait *= 2
            elif status == 500:
                print(f"❌ Server error (500) for {subgroup_id} — skipping this subgroup.")
                return []
            else:
                print(f"❌ HTTP {status} for {subgroup_id}: {e}")
                return []
        except Exception as e:
            print(f"⚠️ Error fetching {subgroup_id}: {e}, retrying in {wait}s...")
            time.sleep(wait)
            retries += 1
            wait *= 2

    print(f"❌ Failed to fetch {subgroup_id} after {MAX_RETRIES} retries — skipping.")
    return []

for subgroup_id in df_subgroups['subgroup_id'].dropna():
    if str(subgroup_id).startswith("EK"):
        continue

    json_filename = f"../data/classgroups/{subgroup_id}.json"

    if os.path.exists(json_filename):
        with open(json_filename, "r") as f:
            students = json.load(f).get("students", [])
    else:
        print(f"Fetching data for subgroup {subgroup_id}")
        students = fetch_students_with_retry(subgroup_id, token)
        if students:
            with open(json_filename, "w") as f:
                json.dump({"students": students}, f, indent=4)
        # else:
        #     
        #     with open(json_filename, "w") as f:
        #         json.dump({"students": []}, f)

#     all_data_json["students"].extend(students)

# combined_json_path = "../data/classgroups/all_students.json"
# with open(combined_json_path, "w") as f:
#     json.dump(all_data_json, f, indent=4)

#TODO: get new token upon expiration time
print("✅")

In [ ]:
import json
import os

input_folder = "../data/classgroups"
output_file = "../data/classgroups/all_students.json"

all_students = []

def extract_students(data):
    if isinstance(data, list):
        if data and isinstance(data[0], dict) and "EMAIL" in data[0]:
            return data
        return []
    if isinstance(data, dict):
        for key, value in data.items():
            if key.strip().lower() == "students":
                if isinstance(value, list):
                    return value
                elif isinstance(value, dict):
                    inner = extract_students(value)
                    if inner:
                        return inner
        for v in data.values():
            inner = extract_students(v)
            if inner:
                return inner
    return []


for filename in os.listdir(input_folder):
    if not filename.endswith(".json"):
        continue

    path = os.path.join(input_folder, filename)
    try:
        with open(path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)
    except Exception as e:
        print(f"⚠️ Skipping {filename}: {e}")
        continue

    students = extract_students(data)
    if students:
        print(f"✅ {filename}: found {len(students)} students")
        all_students.extend(students)
    else:
        print(f"❌ {filename}: no students found")

print(f"\n📊 Total students collected: {len(all_students)}")

# ✅ keep everyone, duplicates and all
merged = {"students": all_students}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=4, ensure_ascii=False)

print(f"💾 Saved {len(all_students)} students to {output_file}")


In [ ]:
import pandas as pd
# Path to your JSON file
json_file = "../data/classgroups/all_students.json"  # replace with your actual file

# Load JSON data
with open(json_file, "r") as f:
    data = json.load(f)

# Extract the 'students' list
students_list = data.get("students", [])

# Convert to DataFrame
students_df = pd.DataFrame(students_list)


students_df.to_csv("../data/all_students.csv", index=False)

students_df.to_csv("../data/classgroups/all_students.csv", index=False)

print(students_df.head())

In [ ]:
students_df.head(10)